In [ ]:
import pandas as pd
import numpy as np

In [ ]:
da = pd.read_csv('train.csv')

In [ ]:
da.head()

In [ ]:
da.drop(columns = ['PassengerId' , 'Name' , 'Ticket' , 'Cabin'] , inplace=True )

In [ ]:
da.head()

In [ ]:
from sklearn.model_selection import train_test_split

x_train , x_test , y_train , y_test = train_test_split(da.drop(columns = ['Survived']) , da['Survived'] , random_state = 42 , test_size = 0.2) #20%891 in testing - 178

# survived will be in o/p

In [ ]:
x_train.head()

STEP - I

In [ ]:
from sklearn.compose import ColumnTransformer

In [ ]:
# imputation transformer

from sklearn.impute import SimpleImputer

# missing values are in the col embarked & Age also embarked is string so we have to apply the mode - most_frquent

tf1 = ColumnTransformer( [ ('impute_age' , SimpleImputer() , [2] ) , ('impute_embarked' , SimpleImputer(strategy='most_frequent') , [6])  ] , remainder = 'passthrough')

# remainder='passthrough' - means - keep the rest of the columns as they are, otherwise they will be dropped

STEP - II


In [ ]:
 # onehotencoding transformer

 # for embarked and Sex col
from sklearn.preprocessing import OneHotEncoder

tf2 = ColumnTransformer(  [( 'ohe_sex', OneHotEncoder(sparse_output=False , handle_unknown = 'ignore') , [3] ) , ( 'ohe_embarked' , OneHotEncoder(sparse_output=False , handle_unknown = 'ignore') , [1] ) ] , remainder = 'passthrough' )

 # tf2 = ColumnTransformer(  [( 'ohe_sex_embarked', OneHotEncoder(sparse_output=False , handle_unknown = 'ignore') , [1,6] ) ] , remainder = 'passthrough' )


STEP - III

In [ ]:
# MinMaxScaler all features into fixed range
from sklearn.preprocessing import MinMaxScaler

# all the numerical cols are scaled to a range of indices :- [0,1]
tf3 = ColumnTransformer( [ ('scale_cols' , MinMaxScaler() , slice(0,10)) ] ,remainder = 'passthrough' )

# slice means all scaling is appled to cols of indices 0-9 , two cols of sex and three of embarked so , total10

STEP IV

In [ ]:
# feature selection

from sklearn.feature_selection import SelectKBest , chi2
tf4 = SelectKBest(score_func= chi2 , k=8)

# method to select the top k features based on a scoring function
 # score_func= chi2 -:Chi-squared (??) statistical test to score features only works on non neg value

STEP V

In [ ]:
from sklearn.tree import DecisionTreeClassifier
tf5 = DecisionTreeClassifier()

PIPELINES

In [ ]:
from sklearn.pipeline import Pipeline
# Pipeline is used to chain multiple steps (transformers + model) together in ml workflow

pipe = Pipeline( [ ('trf1' , tf1) , ('trf2' , tf2) , ('trf3' , tf3) , ('trf4' , tf4) , ('trf5' , tf5) ] )

 # Each step is a tuple -
  # The first value is the name you assign to that step (string).
# The second value is the transformer or estimator object


In [ ]:
# alternative synatx as deos not need to name the steps same as in make_column_transformer
 #from sklearn.pipeline import make_pipeline
#pipe = make_pipeline(tf1 , tf2 , tf3 , tf4 , tf5)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
# train
pipe.fit(x_train,y_train)

# The pipeline applies all the transformers to x_train and y_train remain unchanged
 # LogisticRegression gets the processed X_train and the original y_train to learn the mapping.

EXPLORING THE PIPELINE

In [ ]:
pipe.named_steps        # in key value form

In [ ]:
pipe.named_steps['trf1'].transformers_[0]

In [ ]:
pipe.named_steps['trf1'].transformers_[0][1]       # here u can see that in the tuple 0'thindex of 1

In [ ]:
pipe.named_steps['trf1'].transformers_[0][1].statistics_

In [ ]:
# for embarked we can see
pipe.named_steps['trf1'].transformers_[1][1].statistics_

In [ ]:
y_pred = pipe.predict(x_test)

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score( y_test , y_pred )